# 06. Agent Harness Patterns — Deep Agents에서 유스케이스로 넘어가는 다리

> **왜 이 노트북이 필요한가요?**
>
> 앞의 01~05번 노트북은 Deep Agents의 기능을 하나씩 배웠어요. 그런데 실제 프로젝트에서는 "기능 목록"보다 **어떤 패턴을 언제 선택할지**가 더 중요해요. 이 노트북은 Part 10에서 배운 하네스 능력을 Part 11의 실전 유스케이스로 연결하는 징검다리예요.

> 🔑 **핵심 관점**: Deep Agents는 단순히 `create_deep_agent` 함수 하나가 아니라, 장시간 작업을 위한 **하네스 설계 철학**이에요. 계획을 세우고, 컨텍스트를 밖으로 빼고, 전문 작업을 격리하고, 산출물을 파일로 남기며, 필요할 때 사람의 승인을 받도록 만드는 방식입니다.


## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `create_agent`, 직접 `StateGraph`, `create_deep_agent`의 선택 기준을 설명할 수 있어요
2. Deep Agents의 하네스 능력을 major agent-building pattern 관점으로 재해석할 수 있어요
3. Part 11의 Plan-and-Execute, Deep Research, Data Analysis, Three-Agent 패턴이 Part 10과 어떻게 연결되는지 설명할 수 있어요
4. 새 유스케이스를 만들 때 "원리 학습용 그래프"와 "프로덕션형 하네스"를 구분해서 설계할 수 있어요


## 사전 지식

- `10_Deep_Agents/01-Deep-Agents-Overview.ipynb`: Deep Agents가 LangGraph 위의 Harness라는 관점
- `10_Deep_Agents/02-Deep-Agent-Capabilities.ipynb`: Planning, Filesystem, Subagents, Context, Code Execution, HITL
- `10_Deep_Agents/03-Context-Engineering.ipynb`: 입력·런타임·압축·격리·장기 메모리
- `10_Deep_Agents/04-Subagents.ipynb`: 전문 서브에이전트 위임
- `10_Deep_Agents/05-Skills-Memory.ipynb`: Skills와 Memory의 progressive disclosure


## 1. 세 가지 구현 레벨

같은 에이전트 문제라도 교육 목적과 운영 목적에 따라 구현 레벨이 달라져요.

| 구현 레벨 | 언제 쓰나요? | 장점 | 주의점 |
|---|---|---|---|
| `create_agent` | 짧은 도구 호출, 단순 ReAct 루프 | 가장 간단하고 빠름 | 긴 작업의 계획·메모리·파일 관리가 약함 |
| 직접 `StateGraph` | 상태, 분기, 재시도 원리를 보여줘야 할 때 | 노드·엣지·상태가 눈에 보임 | 운영용 하네스를 직접 많이 만들어야 함 |
| `create_deep_agent` | 긴 작업, 파일 산출물, 위임, 컨텍스트 관리가 필요할 때 | planning/filesystem/subagents/context가 기본 제공 | 내부 하네스가 대신 해주는 일을 먼저 이해해야 함 |

> 🎯 **강의 포인트**: Part 11의 많은 노트북은 먼저 `StateGraph`로 원리를 분해해서 보여주고, 보강 섹션에서 `create_agent` 또는 `create_deep_agent`로 실무형 baseline을 제시해요. 두 방식은 경쟁 관계가 아니라 **교육용 분해도와 운영용 하네스**의 관계입니다.


## 2. Deep Agents 하네스 철학을 5대 패턴으로 보기

공식 문서의 capability 목록은 기능별 분류예요. 수업에서는 이것을 다음 5대 설계 패턴으로 묶어 설명하면 Part 11과 자연스럽게 이어집니다.

| 패턴 | Deep Agents 능력 | 해결하는 문제 | Part 11 연결 |
|---|---|---|---|
| **Plan** | `write_todos`, task tracking | 긴 작업을 한 번에 처리하려다 빠뜨림 | `01-Plan-And-Execute` |
| **Externalize** | filesystem, backend, memory | 대화창에 모든 중간 결과를 쌓아 context bloat 발생 | `08-Data-Analysis-Agent` |
| **Delegate** | subagents, task tool, async subagents | 한 에이전트가 모든 전문 작업을 떠안음 | `03-Deep-Research-Agent`, `09-Three-Agent-Pattern` |
| **Isolate** | fresh subagent context, concise final report | 도구 출력과 세부 추론이 메인 컨텍스트를 오염 | Research/Data/Reviewer 역할 분리 |
| **Verify** | HITL, evaluator, tests, structured artifacts | 에이전트가 스스로 완료했다고 착각 | `09-Three-Agent-Pattern`, Part 12 Testing |

```mermaid
flowchart LR
    USER["사용자 목표"] --> PLAN["Plan
write_todos / spec"]
    PLAN --> EXT["Externalize
files / artifacts"]
    EXT --> DEL["Delegate
subagents"]
    DEL --> ISO["Isolate
fresh context"]
    ISO --> VER["Verify
HITL / evaluator / tests"]
    VER --> RESULT["검증된 산출물"]
    VER -->|"피드백"| PLAN
```

> 🔑 **핵심 개념**: 좋은 에이전트 하네스는 모델을 더 세게 몰아붙이는 장치가 아니라, 모델이 오래 일해도 덜 흐트러지도록 **작업 환경을 설계하는 장치**예요.


## 3. Part 11 노트북을 Part 10 관점으로 다시 읽기

Part 11은 "새 개념을 마구 추가하는 장"이 아니라, Part 10까지 배운 agent-building pattern을 도메인 문제에 적용하는 장이에요.

| Part 11 노트북 | 먼저 배우는 원리 | Deep Agents 관점의 확장 |
|---|---|---|
| `01-Plan-And-Execute` | Planner → Executor → Replanner | `write_todos`와 filesystem으로 장기 작업 하네스화 |
| `03-Deep-Research-Agent` | STORM, map-reduce, HITL | researcher subagent, context isolation, report artifact |
| `08-Data-Analysis-Agent` | 파일 탐색 → 분석 → 시각화 → 보고서 | FilesystemBackend와 artifact loop의 대표 사례 |
| `09-Three-Agent-Pattern` | Planner / Generator / Evaluator | Delegate + Verify를 명시한 long-running harness 패턴 |

> 💡 **운영 팁**: 비전공자 반에서는 Part 11 전체를 같은 깊이로 다루기보다, 이 표를 먼저 보여준 뒤 SQL, Research, Data Analysis 중 하나를 선택 실습으로 운영하는 편이 좋아요.


## 4. 선택 기준: 직접 그래프인가, Deep Agent인가?

아래 체크리스트로 구현 방식을 고르면 됩니다.

| 질문 | Yes라면 |
|---|---|
| 노드·엣지·상태 전이를 학생이 눈으로 이해해야 하나요? | 직접 `StateGraph` |
| 실패 시 재시도 조건을 코드로 엄격히 제어해야 하나요? | 직접 `StateGraph` 또는 hybrid |
| 중간 산출물이 파일/리포트/차트처럼 남아야 하나요? | `create_deep_agent` |
| 작업이 10분 이상 길어지고 컨텍스트가 커지나요? | `create_deep_agent` |
| 리서처·작성자·리뷰어처럼 역할 전문화가 필요한가요? | `create_deep_agent(subagents=...)` |
| 사람이 승인해야 하는 위험한 도구 호출이 있나요? | `create_deep_agent(interrupt_on=...)` |

> ⚠️ **자주 하는 실수**: `StateGraph`로 원리를 배운 뒤 모든 실전 문제를 직접 그래프로만 풀려고 하면, planning·memory·filesystem·HITL 같은 운영 하네스를 매번 다시 만들어야 해요. 반대로 처음부터 `create_deep_agent`만 쓰면 상태 전이와 실패 복구 원리를 놓칠 수 있어요.


In [ ]:
# 선택 기준을 코드로 표현한 간단한 의사결정 예시입니다.
# 실제 프로젝트에서는 보안, 비용, 배포 환경까지 함께 고려하세요.

def choose_agent_implementation(
    *,
    needs_visible_state_graph: bool,
    needs_strict_retry_control: bool,
    long_running: bool,
    produces_artifacts: bool,
    needs_specialist_roles: bool,
    needs_human_approval: bool,
) -> str:
    """학습/운영 요구에 맞는 에이전트 구현 레벨을 추천해요."""
    if needs_visible_state_graph or needs_strict_retry_control:
        if long_running or produces_artifacts or needs_specialist_roles:
            return "hybrid: StateGraph로 핵심 제어를 보이고, 보강에서 create_deep_agent 하네스를 제시"
        return "direct StateGraph"

    if long_running or produces_artifacts or needs_specialist_roles or needs_human_approval:
        return "create_deep_agent"

    return "create_agent"


choose_agent_implementation(
    needs_visible_state_graph=True,
    needs_strict_retry_control=True,
    long_running=True,
    produces_artifacts=True,
    needs_specialist_roles=True,
    needs_human_approval=False,
)


## 5. 정리: Part 10과 Part 11의 경계

이제 두 파트의 역할을 이렇게 잡으면 됩니다.

| 파트 | 역할 | 학생이 가져가야 할 문장 |
|---|---|---|
| Part 10 Deep Agents | 장기 실행 에이전트의 하네스 철학과 구성 능력 | "긴 작업은 계획·파일·위임·격리·검증 환경을 먼저 설계한다" |
| Part 11 Use Cases | 도메인별 문제에 패턴을 적용하는 응용 장 | "SQL, Research, Data Analysis, GraphRAG는 같은 패턴의 다른 얼굴이다" |

따라서 Part 11에 있는 `create_deep_agent` 보강 섹션은 Part 10과 중복되는 내용이 아니라, **Part 10의 철학을 실제 유스케이스에 다시 꽂아보는 연결부**로 읽으면 좋아요.


## 다음 노트북 예고

다음 `11_Use_Cases/01-Plan-And-Execute.ipynb`에서는 이 노트북의 **Plan** 패턴을 직접 `StateGraph`로 분해해요. 먼저 Planner / Executor / Replanner의 원리를 눈으로 확인한 뒤, 보강 섹션에서 같은 아이디어를 `create_deep_agent` 하네스로 확장하는 방법을 비교합니다.
